<a href="https://colab.research.google.com/github/jccrews256/ST-554-HW-8/blob/main/Homework_8_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ST 554 Homework 7
*By: Cass Crews*

In this notebook, we will explore the process of developing machine learning models to predict future values of some response variable of interest. In particular, we will fit a final predictive model for both a regression problem and a classification problem while considering multiple model classes for each.

While the goal of the notebook is the process of developing a predictive model rather than to actually develop a model for some use case, we still need an example dataset. We will use the [Wine Quality dataset](https://archive.ics.uci.edu/dataset/186/wine+quality) from the UC-Irvine Machine Learning Repository. This dataset documents several characteristics of nearly 7,000 wines from northern Portugal.

For the regression task, our goal is to construct a model that adequately predicts the alcohol level of a wine given its other characteristics. We will consider the following model classes:
- Multiple linear regression models
- LASSO regression models
- Ridge regression models
- Elastic net regression models
- Regression tree models
- Random forest regression models.

To obtain the final model, will first split our dataset of wines into a training and test set. We will then use 10-fold cross-validation on the training set to identify the best model from each class. The final step will be to use the top-performing model from each class to predict alcohol content for the test set, allowing us to identify the best-performing overall model.

For the classification task, our goal is to construct a model that adequately predicts whether a wine is red or white given its other characteristics. In this case, we will consider:
- Logistic regression models
- Logistic regression models with L1 penalty
- Logistic regression models with L2 penalty
- Logistic regression models with elastic net penalty
- Classification tree models
- Random forest classification models.

Note that while logistic regression includes the term "regression," each of the four classes above are classification models from the machine learning perspective. That is, the response is categorical rather than a continuous numeric variable. The steps to obtain a final classification model will be the same steps used for the regression problem.

## Preparatory Steps

Before we begin tuning models, we need to read in the necessary modules and data. We'll start with the modules:

In [9]:
import pandas as pd
import numpy as np
from sklearn import linear_model
from math import sqrt
from sklearn.model_selection import train_test_split, cross_validate, KFold, \
    GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, log_loss, \
    accuracy_score
from sklearn.linear_model import LinearRegression, LassoCV, Lasso, RidgeCV, \
    Ridge, ElasticNetCV, ElasticNet, LogisticRegression, LogisticRegressionCV
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import PolynomialFeatures

We're now ready to read in the data. The white and red wines are stores in separately csv files, so we will combine both wine types into a single `pandas` dataframe. In doing so, we will create a dummy variable that indicates whether each wine is `red`, with a value of 1 if so and 0 if not.

*Note to the reader: You will need to store the two data files in your local environment to run the remaining code chunks in this notebook.*

In [10]:
# Reading in the red wine data and adding red indicator
red_wines = pd.read_csv("winequality-red.csv", sep = ";")
red_wines["red"] = 1

# Reading in the white wine data and adding red indicator
white_wines = pd.read_csv("winequality-white.csv", sep = ";")
white_wines["red"] = 0

# Concatenating the two dataframes
wines = pd.concat([red_wines, white_wines], ignore_index = True)

Let's figure out how many red and white wines are in the dataset.

In [11]:
wines.red.value_counts()

,count
red,
0,4898
1,1599


There are roughly three times as many white wines as red wines. This is something we'll need to keep in mind when we split the data into training and test sets, as this class imbalance could lead to us accidentally creating a training set with predominantly white wines.

Let's print the first few rows to see the candidate predictors available to us for our modeling problems.

In [12]:
# Extracting first five rows
wines.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,red
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,1
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,1
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,1
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,1
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,1


we have a variety of characteristics for each wine, ranging from `fixed acidity` to `density` and sulphate levels.

An immediate concern for this dataframe is that the variable names include spaces. This can make it difficult to extract individual variables, so let's replace each space with an underscore.

In [13]:
# Replacing spaces with underscores in column names
wines.columns = [col.replace(" ","_") for col in wines.columns.to_list()]

# Extracting first few rows
wines.head()

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,quality,red
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,1
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,1
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,1
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,1
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,1


These names are much more manageable.

The last step before we move to the regression problem is to confirm there are no missing values for any of the columns.

In [14]:
# Checking for missing values
wines.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6497 entries, 0 to 6496
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed_acidity         6497 non-null   float64
 1   volatile_acidity      6497 non-null   float64
 2   citric_acid           6497 non-null   float64
 3   residual_sugar        6497 non-null   float64
 4   chlorides             6497 non-null   float64
 5   free_sulfur_dioxide   6497 non-null   float64
 6   total_sulfur_dioxide  6497 non-null   float64
 7   density               6497 non-null   float64
 8   pH                    6497 non-null   float64
 9   sulphates             6497 non-null   float64
 10  alcohol               6497 non-null   float64
 11  quality               6497 non-null   int64  
 12  red                   6497 non-null   int64  
dtypes: float64(11), int64(2)
memory usage: 660.0 KB


It seems there are no missing values. The dataset web page (linked above) also indicates there are no missing values, but it's always best to check!

## Regression Task

We are now ready to develop a prediction model for `alcohol` level. As we discussed in the introduction, we will consider candidate models across four different model classes:
- Multiple linear regression models
- LASSO regression models
- Ridge regression models
- Elastic Net regression models.

#### Preparing Test and Training Sets

The first step is to separate the full `wines` dataset into training and test sets. We will reserve 20\% for the test set and the rest for the training set. As there are many more white wines than red wines and we don't want to accidentally train our models using predominantly white wines, we will use stratified sampling to ensure the proportions for each wine type are roughly equal across training and test sets.

In [15]:
# Splitting into training and test sets
# Stratifying based on red (wine type)
X_train, X_test, y_train, y_test = train_test_split(
    wines.drop("alcohol", axis = 1),
    wines.alcohol,
    test_size = 0.20,
    stratify = wines.red,
    random_state = 5
)

As three of the model lasses we will consider involve coefficient shrinkage, we need to standardize each predictor to ensure differences in scale across predictors are not influencing the relative shrinkage of the predictors' coefficient estimates. We will save the predictor means and standard deviations from the training set so that we can apply the same standardizing transformations to the test set. This ensures the training and test sets are on the same scale.

In [16]:
# Capturing means and standard deviations for predictors
means = X_train.apply(np.mean, axis = 0)
stds = X_train.apply(np.std, axis = 0, ddof = 1) # n-1 degrees of freedom

Now that we've saved the training set means and standard deviations, let's standardized the training set predictors.

In [17]:
# Standardizing the training set predictors
X_train = X_train.apply(lambda x: (x - np.mean(x, axis = 0))/
                        np.std(x, axis = 0, ddof = 1))

Let's now transform the test set predictors using the training set means and standard deviations.

In [18]:
# Creating basic function to standardize using supplied mean and standard dev.
def std_col(col, mean, std):
    return (col-mean)/std

# Loop through predictors and standardize for test set
for c in X_test.columns:
    X_test[c] = std_col(X_test[c], means[c], stds[c])

To highlight the fact that we are standardizing candidate predictors in the training and test sets using training set means and standard deviations, we can extract the variables' rescaled means for each set.

In [19]:
pd.DataFrame([X_train.mean(), X_test.mean()], index = ["train_mean", "test_mean"])

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,quality,red
train_mean,2.187547e-16,4.648538e-17,1.093774e-16,-3.623125e-17,9.707242e-17,-4.511816e-17,-1.039085e-16,1.119477e-14,1.467708e-15,5.468868e-18,-1.702185e-16,3.486404e-17
test_mean,-3.063200e-02,-3.260071e-02,1.552467e-03,-3.463125e-02,-1.962903e-02,3.663746e-02,-2.107540e-02,-4.478241e-02,3.245664e-02,-6.490676e-03,3.659459e-02,1.168224e-04


Note that the training set means are all effectively zero, while the test set means are close to zero but not effectively zero.

#### Tuning Multiple Linear Regression

We are now ready to tune the multiple linear regression (MLR) model class to identify the "best" model for predicting `alcohol`. To do so, let's consider four different parametrizations:
- $E(\text{alcohol}|X) = \beta_0 + \beta_1\text{red} + \beta_2\text{sulphates} + \beta_3\text{residual_sugar}$
- $E(\text{alcohol}|X) = \beta_0 + \beta_1\text{red} + \beta_2\text{sulphates} + \beta_3\text{residual_sugar} + \beta_4\text{pH}$
- $E(\text{alcohol}|X) = \beta_0 + \beta_1\text{red} + \beta_2\text{sulphates} + \beta_3\text{residual_sugar} + \beta_4\text{pH} + \beta_5\text{red} \cdot \text{sulphates} + \beta_6\text{red} \cdot \text{residual_sugar} + \beta_7\text{red} \cdot \text{pH}$
- $E(\text{alcohol}|X) = \beta_0 + \beta_1\text{red} + \beta_2\text{sulphates} + \beta_3\text{residual_sugar} + \beta_4\text{pH} + \beta_5\text{red} \cdot \text{sulphates} + \beta_6\text{red} \cdot \text{residual_sugar} + \beta_7\text{red} \cdot \text{pH} + \beta_8\text{sulphates}^2 + \beta_9\text{residual_sugar}^2 + \beta_{10}\text{pH}^2$

Why have I chosen these models? Because I actually know what `sulphates`, `residual_sugar`, and `pH` mean in the context of wine, something I cannot say for many of the other candidate predictors....

Conceptually, what are we considering? We are effectively considering whether the linear effects of `sulphates`, `residual_sugar`, and `pH` vary across wine types, as well as whether these variables have quadratic effects that are not conditional on wine type. Of course, we are simply predicting `alcohol` content, so our only "test" for such relationships will be whether they improve our ability to predict.

Before we generate the cross-validated root mean squared errors (RMSEs) for these models to select the best one for predicting `alcohol` level, we need to construct the interaction and quadratic terms for models 3 and 4. As we will need to generate these same terms for the test set, let's build some helper functions. We'll use the helper functions to add the new terms to `X_train` and print the first few rows to ensure our helpers work.

In [20]:
# Creating interative term helper function that adds quadratic terms and
# returns updated df
def int_maker(df, col1, col2):
    df1 = df.copy() # Making copy of df for modification
    df1[f"{col1}_{col2}"] = df1[col1] * df1[col2]
    return df1

# Creating quadratic term helper function that modifies and returns df
def quad_maker(df, cols):
    df1 = df.copy() # Making copy of df for modification
    # Squaring each element in cols
    for c in cols:
        df1[f"{c}_sq"] = df1[c]**2
    return df1

# Copying training design matrix for MLR
X_train_MLR = X_train.copy()

# Creating interaction terms for models 3 and 4
for c in ["sulphates", "residual_sugar", "pH"]:
    X_train_MLR = int_maker(X_train_MLR, "red", c)

# Creating quadratic terms for model 4
X_train_MLR = quad_maker(X_train_MLR, ["sulphates", "residual_sugar", "pH"])

# Confirming variables were created
X_train_MLR.head()

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,quality,red,red_sulphates,red_residual_sugar,red_pH,sulphates_sq,residual_sugar_sq,pH_sq
3181,0.894526,-0.608712,1.168502,0.400502,-0.489518,0.893113,-0.141267,-0.439097,-1.161098,-1.347966,1.365426,-0.571296,0.770088,-0.228805,0.663331,1.817012,0.160402,1.348149
1703,0.134282,-0.548291,0.350473,1.670625,0.109136,1.236557,1.345220,0.921008,-1.346921,-0.611966,-0.933285,-0.571296,0.349614,-0.954421,0.769491,0.374503,2.790987,1.814196
4485,-0.473913,0.418440,-0.058541,-0.807156,-0.403996,-0.709628,-0.265141,-0.920110,-0.541688,-0.812693,-0.933285,-0.571296,0.464289,0.461125,0.309465,0.660470,0.651500,0.293426
6421,-0.321864,-0.669132,1.100333,-0.827977,-0.575040,0.263465,0.867420,-1.029581,-0.231984,-0.545057,0.216071,-0.571296,0.311389,0.473020,0.132531,0.297087,0.685547,0.053816
1921,-0.321864,-0.548291,0.145966,-0.161684,-0.717576,0.206224,0.318836,-0.671310,-0.046161,-1.013420,0.216071,-0.571296,0.578963,0.092369,0.026371,1.027021,0.026142,0.002131


We are now ready to generate cross-validated RMSEs for each model! Note: we will not use the quadratic and interaction terms in the regularized regression models, so we do not need to standardize them.

In [21]:
# Applying CV to first MLR model
mlr_cv1 = cross_validate(LinearRegression(),
                     X_train_MLR[["red", "sulphates", "residual_sugar"]],
                     y_train,
                     cv = 10,
                     scoring = 'neg_mean_squared_error')

# Applying CV to second model
mlr_cv2 = cross_validate(LinearRegression(),
                     X_train_MLR[["red", "sulphates", "residual_sugar", "pH"]],
                     y_train,
                     cv = 10,
                     scoring = 'neg_mean_squared_error')

# Applying CV to third model
mlr_cv3 = cross_validate(LinearRegression(),
                     X_train_MLR[["red", "sulphates", "residual_sugar", "pH",
                                    "red_sulphates", "red_residual_sugar",
                                    "red_pH"]],
                     y_train,
                     cv = 10,
                     scoring = 'neg_mean_squared_error')

# Applying CV to fourth model
mlr_cv4 = cross_validate(LinearRegression(),
                     X_train_MLR[["red", "sulphates", "residual_sugar", "pH",
                                    "red_sulphates", "red_residual_sugar",
                                    "red_pH", "sulphates_sq", "residual_sugar_sq",
                                    "pH_sq"]],
                     y_train,
                     cv = 10,
                     scoring = 'neg_mean_squared_error')

# Printing RMSE values
print(round(sqrt(-np.mean(mlr_cv1["test_score"])),4),
      round(sqrt(-np.mean(mlr_cv2["test_score"])),4),
      round(sqrt(-np.mean(mlr_cv3["test_score"])),4),
      round(sqrt(-np.mean(mlr_cv4["test_score"])),4))

1.1022 1.0991 1.0905 1.0976


It seems our third model, which includes the interactions but not the quadratic terms, produces the lowest cross-validated error. Let's fit our best model to the full training set.

In [22]:
# Fitting best model to full training set
mlr_best = LinearRegression().fit(X_train_MLR[["red", "sulphates", "residual_sugar", "pH",
                                "red_sulphates", "red_residual_sugar",
                                "red_pH"]],
                     y_train)

We now have a model we can use to predict `alcohol` for the test set. However, we need to add the interaction terms to `X_test` in order to do so.

In [23]:
# Adding quadratic terms to the test set
# Copying training design matrix for MLR
X_test_MLR = X_test.copy()

# Creating interaction terms for models 3 and 4
for c in ["sulphates", "residual_sugar", "pH"]:
    X_test_MLR = int_maker(X_test_MLR, "red", c)

We now have everything we need to predict `alcohol` in the test set using our best MLR model.

#### Tuning LASSO Regression

We are now ready to tune a LASSO model for predicting `alcohol`. To prevent our design matrix from becoming excessively large, we will not consider any interaction or quadratic terms for this model or any other regularization model. Instead, we will consider all individual candidate predictors in the dataset except `quality`; `quality` is a subjective rating that may be difficult to collect for future wines we will predict `alcohol` content for.

Let's find the optimal value of $\alpha$, the tuning parameter for the LASSO model! We will consider 100 values between $10^{-4}$ and $10^{2}$.

In [24]:
# Using CV to optimize alpha in LASSO model
lasso_model = LassoCV(cv = 10, random_state = 5, alphas = np.logspace(-4, 2, 100)) \
    .fit(X_train.drop("quality", axis = 1), y_train)

# Printing the optimal alpha
print(lasso_model.alpha_)

0.0004641588833612782


That's a very small value for $\alpha$, indicating our fit will be close to the OLS fit for a model with all predictors excluding `quality`. Let's use this "best" value to fit a LASSO model to the full training set.

In [25]:
# Fitting best LASSO model to full training set
lasso_best = Lasso(lasso_model.alpha_) \
.fit(X_train.drop("quality", axis = 1), y_train)

We now have a tuned LASSO model to predict `alcohol` for the test set.


#### Tuning Ridge Regression

Let's repeat the LASSO-tuning process for the ridge regression model class. Here, we are tuning $\lambda$, although `RidgeCV` confusingly calls it alpha. We will consider the same 100 values considered for $\alpha$ above.

In [26]:
# Using CV to optimize "alpha" (lambda) in Ridge model
ridge_model = RidgeCV(cv = KFold(n_splits = 10, shuffle=True, random_state = 5), \
                      alphas = np.logspace(-4, 2, 100)) \
    .fit(X_train.drop("quality", axis = 1), y_train)

# Printing the optimal alpha (lambda)
print(ridge_model.alpha_)

6.135907273413176


$\lambda$ is not near 0, potentially indicating the design matrix suffers from multicollinearity in the predictors. However, the scale of the LASSO and ridge penalty terms are not equivalent, so it is difficult to draw conclusions based on hyperparmaeter values alone.

Let's use our best value of $\lambda$ to fit a ridge regression model to the full training set.

In [27]:
# Fitting best ridge model to the full training set
ridge_best = Ridge(ridge_model.alpha_) \
.fit(X_train.drop("quality", axis = 1), y_train)

We now have a tuned ridge regression model to predict `alcohol` for the test set. Only one more model class to go!

#### Tuning Elastic Net Regression

We are now ready to tune the elastic net regression model class. As the elastic net objective function incorporates both the LASSO and Ridge penalty terms, we need to tune two separate hyperparameters. `ElasticNetCV` refers to these as `l1_ratio` and `alpha`, but we are effectively tuning $\lambda$ and $\alpha$, respectively; `l1_ratio` captures the relationship between $\lambda$ and $\alpha$ with values near 0 indicating a near-ridge fit and values near 1 indicating a near LASSO fit. To keep the computation manageable, we'll consider the same 100 `alpha` values we considered for the LASSO model and a relatively sparse grid of `l1_ratio` values ranging from 0.1 to 1.

In [28]:
# Using CV to tune l1_ratio and alpha for elastic net
en_model = ElasticNetCV(cv = 10, random_state = 5,
                        l1_ratio = [0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.965, 0.98, 0.99, 1],
                        alphas = np.logspace(-4, 2, 100)) \
.fit(X_train.drop("quality", axis = 1), y_train)

# Printing optimal parameters
print(en_model.alpha_, en_model.l1_ratio_)

0.0004641588833612782 0.99


Interestingly, the $\alpha$ value selected by cross validation is the same small value for the LASSO model and `l1_ratio` is nearly 1; it seems our model does not warrant any regularization! Let's use the best combination of tuning parameters to fit an elastic net model to the full training set.

In [29]:
# Fitting best elastic net model to the full training set
en_best = ElasticNet(alpha = en_model.alpha_, l1_ratio = en_model.l1_ratio_) \
.fit(X_train.drop("quality", axis = 1), y_train)

We now have all the tuned models we need to compare our four model classes' ability to predict `alcohol` for the test set!

#### Tuning Regression Tree

Up to this point, we have used the linear regression model and its regularized extensions to predict `alcohol` level. It may be the case that the relationship between `alcohol` and the predictors is more complex (non-linear) than those model classes can effectively provide. Thus, tree-based models could be more suitable for this problem. We will now consider such a model class: the regression tree. A regression tree is a single tree that splits the model space into multiple regions based on the values of predictors. The ability to sequentially split one "branch" of the tree by multiple predictors inherently allows for complex non-linear relationships.

Tuning this model involves tuning two hyperparameters:
- `max_depth` sets the maximum number of splits a single branch can have, ensuring we don't create trees with more complex relationships than necessary.
- `min_samples_leaf` sets the minimum number of training set observations that can be in a single leaf (end node), ensuring we don't overfit the training set even for niche corners of the model space.

Let's tune a regression tree by considering tree depths ranging from 2 to 18 splits and minimum leaf observations ranging from 3 to 100.


In [39]:
# Specifying candidate hyperparameter values
parameters = {'max_depth': range(2,19),
    'min_samples_leaf': [3, 5, 10, 25, 50, 100]}

# Tuning tree using 10-fold CV
rtree_model = GridSearchCV(DecisionTreeRegressor(random_state = 5),
    parameters,
    cv = KFold(n_splits = 10, shuffle=True,
        random_state = 5),
    scoring = "neg_mean_squared_error") \
    .fit(X_train.drop("quality", axis = 1), y_train)

# Printing optimal parameters
print(rtree_model.best_estimator_)


DecisionTreeRegressor(max_depth=15, min_samples_leaf=5, random_state=5)


The best-performing model is deep (up to 15 splits per branch) and has some "small" leaves -- leaves can have as few as 5 training observations.

To compare our best regression tree model to the best models from other classes, we need to fit a tree to the full training set using the best-performing hyperparameters. Conveniently, `GridSearchCV` already did that for us, so we can simply capture this fitted model.

In [43]:
# Capturing regression tree fit to full training set using best hyperparameters
rtree_best = rtree_model.best_estimator_

We have everything we need to compare our best regression tree models to the best models from the other classes.

#### Tuning Random Forest

A limitation of the regression tree is that a single tree can provide a highly variable fit across training sets, often resulting in individual models that don't generalize well to new data. The random forest model is one solution to this issue. It creates many trees by taking random samples with replacement from the training set and fitting a tree to each sample. Predictions for random forest regression models are then the average prediction across the many trees. This reduces the variability of model fit, given hyperparameter values, across training sets.

For the random forest, the main tuning parameter, `max_features`, is the number of predictors randomly selected for consideration at each tree split. In many ways, this is what makes the random forest a powerful prediction tool. If every predictor were available to the model at each split, a few predictors would dominate the main (initial) splits for each tree. This would make the trees highly correlated and result in minimal variance-reduction benefits from fitting many trees.

We can also tune the hyperparameters we tuned for the regression tree model class: `max_depth` and `min_samples_leaf`. We need to set one more hyper parameter: `n_estimators` determines the number of trees fit. It is common to simply set this hyperparameter at a high value, such as 500 or 1,000, rather than tuning the number of trees. We'll fit 500 trees per model.

Let's tune the random forest model! We'll consider `max_features` values ranging from 1 to 11 (total number of predictors), `max_depth` values ranging from 3 to 18, and `min_samples_leaf` ranging from 3 to 100. We are fitting a lot of trees, so we'll utilize the parallel computing capabilities of `sklearn` by passing `n_jobs = -1` to `GridSearchCV`. This results in the function using all but one of the cores available.

In [49]:
# Specifying candidate hyperparameter values
parameters = {'max_features': range(1,12),
              'max_depth': [3, 5, 10, 15, 18, 21, 24],
    'min_samples_leaf': [3, 5, 10, 25, 50, 100]}

# Tuning random forest using 10-fold CV
rf_model = GridSearchCV(RandomForestRegressor(n_estimators = 500, random_state = 5),
    parameters,
    cv = KFold(n_splits = 10, shuffle=True,
        random_state = 5),
    scoring = "neg_mean_squared_error",
    n_jobs = -1) \
    .fit(X_train.drop("quality", axis = 1), y_train)

# Printing optimal parameters
print(rf_model.best_estimator_)


RandomForestRegressor(max_depth=18, max_features=11, min_samples_leaf=3,
                      n_estimators=500, random_state=5)


Interestingly, the best-performing value of `max_features` is 10, one below the total number of features. This model is approaching a bagged regression tree model, which is a special case of the random forest where `max_features` is exactly equal to the number of predictors. Another interesting fact is that these trees are deep (up to 15 splits per branch) and can have very small leaves (as few as 3 training observations per leaf).

As we have again used `GridSearchCV` to tune the model, we already have a model with the best-performing hyperparameters fit to the full training set. Let's capture it.

In [52]:
# Capturing random forest fit to full training set using best hyperparameters
rf_best = rf_model.best_estimator_

#### Choosing a Final Model to Predict `alcohol`

Now that we have tuned each model class on our training set, we are ready to predict `alcohol` for the test set using the best model from each class. We'll calculate the resultant test RMSE and mean absolute error (MAE) for each model to select a best overall predictive model.  

In [53]:
# Predicting with MLR model
mlr_pred = mlr_best.predict(X_test_MLR[["red", "sulphates", "residual_sugar", "pH",
                                    "red_sulphates", "red_residual_sugar",
                                    "red_pH"]])

# Predicting with LASSO model
lasso_pred = lasso_best.predict(X_test.drop("quality", axis = 1))

# Predicting with Ridge model
ridge_pred = ridge_best.predict(X_test.drop("quality", axis = 1))

# Predicting with Elastic Net model
en_pred = en_best.predict(X_test.drop("quality", axis = 1))

# Predicting with regression tree model
rtree_pred = rtree_best.predict(X_test.drop("quality", axis = 1))

# Predicting with random forest model
rf_pred = rf_best.predict(X_test.drop("quality", axis = 1))

# Printing RMSE
print("MLR RMSE:", sqrt(mean_squared_error(y_test, mlr_pred)),
      "\nLASSO RMSE:", sqrt(mean_squared_error(y_test, lasso_pred)),
      "\nRidge RMSE:", sqrt(mean_squared_error(y_test, ridge_pred)),
      "\nElastic Net RMSE:", sqrt(mean_squared_error(y_test, en_pred)),
      "\nRegression Tree RMSE:", sqrt(mean_squared_error(y_test, rtree_pred)),
      "\nRandom Forest RMSE:", sqrt(mean_squared_error(y_test, rf_pred)))

# Printing MAE
print("\nMLR MAE:", mean_absolute_error(y_test, mlr_pred),
      "\nLASSO MAE:", mean_absolute_error(y_test, lasso_pred),
      "\nRidge MAE:", mean_absolute_error(y_test, ridge_pred),
      "\nElastic Net MAE:", mean_absolute_error(y_test, en_pred),
      "\nRegression Tree MAE:", mean_absolute_error(y_test, rtree_pred),
      "\nRandom Forest MAE:", mean_absolute_error(y_test, rf_pred))

MLR RMSE: 1.0532454889045075 
LASSO RMSE: 0.45332302428681 
Ridge RMSE: 0.45427782855103915 
Elastic Net RMSE: 0.4533254929946822 
Regression Tree RMSE: 0.5285792758849338 
Random Forest RMSE: 0.38405877097748414

MLR MAE: 0.8572539286312216 
LASSO MAE: 0.3487016231797675 
Ridge MAE: 0.34971347243476175 
Elastic Net MAE: 0.34870448373111895 
Regression Tree MAE: 0.3593717212824493 
Random Forest MAE: 0.26775306191798176


The two test metrics agree that the LASSO model marginally outperforms the ridge and elastic net models in predicting `alcohol`; for both metrics, a lower value is better. In reality, predictive performance is nearly identical.

## Classification Task

Now that we have selected a final model for the regression problem, let's replicate the same tuning and selection process to identify the best predictive model for wine type (`red`). As a reminder, we will consider:
- Logistic regression models
- Logistic regression models with L1 penalty
- Logistic regression models with L2 penalty
- Logistic regression models with elastic net penalty.

We will tune each model class using log-loss as our model metric, and we will consider both log-loss and accuracy when evaluating the performance of the best model from each class on the test set.

#### Preparing Test and Training Sets

The test and training set creation process will be nearly identical to that of the regression problem. The biggest differences is that `red` is now our response variable. We'll again stratify the sampling process by `red` is it is even more important in this case that the wine type distributions are similar across training and test sets. Let's separate the data into training and test sets accordingly.

In [ ]:
# Splitting into training and test sets
# Stratifying based on red (wine type)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    wines.drop("red", axis = 1),
    wines.red,
    test_size = 0.20,
    stratify = wines.red,
    random_state = 5
)

As we are considering three regularization methods, we again need to standardize the predictors. We will again scale the predictors in the test set using their corresponding means and standard deviations in the training set, as this ensures each predictor has the same scale across the sets.

In [ ]:
# Capturing means and standard deviations for predictors in training set
means_c = X_train_c.apply(np.mean, axis = 0)
stds_c = X_train_c.apply(np.std, axis = 0, ddof = 1) # n-1 degrees of freedom

# Standardizing the training set predictors
X_train_c = X_train_c.apply(lambda x: (x - np.mean(x, axis = 0))/
                        np.std(x, axis = 0, ddof = 1))

# Loop through predictors and standardize for test set
for c in X_test_c.columns:
    X_test_c[c] = std_col(X_test_c[c], means_c[c], stds_c[c])

We have completed the training set processing!

#### Tuning Logistic Regression

We are now ready to "tune" the logistic regression model via the training set to identify the best parametrization for predicting wine type. We'll consider the following four parametrizations that, like in the MLR case, rely on wine characteristics I actually understand:

- $\text{log-odds}(\text{red}|X) = \beta_0 + \beta_1\text{alcohol} + \beta_2\text{sulphates}$
- $\text{log-odds}(\text{red}|X) = \beta_0 + \beta_1\text{alcohol} + \beta_2\text{sulphates} + \beta_3\text{residual_sugar}$
- $\text{log-odds}(\text{red}|X) = \beta_0 + \beta_1\text{alcohol} + \beta_2\text{sulphates} + \beta_3\text{residual_sugar} + \beta_4\text{alcohol} \cdot \text{sulphates} + \beta_5\text{alcohol} \cdot \text{residual_sugar} + \beta_6\text{sulphates} \cdot \text{residual_sugar}$
- $\text{log-odds}(\text{red}|X) = \beta_0 + \beta_1\text{alcohol} + \beta_2\text{sulphates} + \beta_3\text{residual_sugar} + \beta_4\text{alcohol} \cdot \text{sulphates} + \beta_5\text{alcohol} \cdot \text{residual_sugar} + \beta_6\text{sulphates} \cdot \text{residual_sugar} + \beta_7\text{alcohol}^2 + \beta_8\text{sulphates}^2 + \beta_9\text{residual_sugar}^2$

Note that the $|X$ in each equation simply indicates the log-odds of a wine being red is conditional on the predictors on the right-hand side (the $x$'s).

Let's generate the average log-loss for each model across the 10 cross-validation folds. Unlike in the regression case, we can simply use `PolynomialFeatures` to generate the interaction and quadratic terms. However, similar to the regression case, we do not need to standardize these new terms because we will not use them in any regularized models.

In [ ]:
# Obtaining CV negative log-loss for model 1
lr_cv1 = cross_validate(LogisticRegression(solver = "newton-cg", penalty = None),
                        X_train_c[["alcohol", "sulphates"]], y_train_c,
                        scoring = "neg_log_loss",
                        cv = 10)

# Obtaining CV negative log-loss for model 2
lr_cv2 = cross_validate(LogisticRegression(solver = "newton-cg", penalty = None),
                        X_train_c[["alcohol", "sulphates", "residual_sugar"]],
                        y_train_c,
                        scoring = "neg_log_loss",
                        cv = 10)

# Obtaining CV negative log-loss for model 3
lr_cv3 = cross_validate(LogisticRegression(solver = "newton-cg", penalty = None),
                        PolynomialFeatures(interaction_only=True, include_bias = False) \
                        .fit_transform(X_train_c[["alcohol", "sulphates", "residual_sugar"]]),
                        y_train_c,
                        scoring = "neg_log_loss",
                        cv = 10)


# Obtaining CV negative log-loss for model 4
lr_cv4 = cross_validate(LogisticRegression(solver = "newton-cg", penalty = None),
                        PolynomialFeatures(interaction_only=False, include_bias = False) \
                        .fit_transform(X_train_c[["alcohol", "sulphates", "residual_sugar"]]),
                        y_train_c,
                        scoring = "neg_log_loss",
                        cv = 10)

# Printing log-loss values
print(round(-np.mean(lr_cv1["test_score"]),4),
      round(-np.mean(lr_cv2["test_score"]),4),
      round(-np.mean(lr_cv3["test_score"]),4),
      round(-np.mean(lr_cv4["test_score"]),4))

There must be a relatively non-linear relationship between the predictors and the log-odds of a wine being red, as the fourth model produces the best (lowest) log-loss. I admit that I did not foresee this outcome.

Let's fit model 4 to the full training set so that we are ready to predict wine type for the test set.

In [ ]:
# Fitting best LR model to full training set
lr_best = LogisticRegression(solver = "newton-cg", penalty = None) \
    .fit(PolynomialFeatures(interaction_only=False, include_bias = False) \
                        .fit_transform(X_train_c[["alcohol", "sulphates", "residual_sugar"]]),
         y_train_c)

We are now ready to predict test set wine types!

#### Tuning Logistic Regression with L1 Penalty

The next model class we need to tune is the logistic regression with an L1 penalty. Similar to the regularized regression models, our regularized classification models will not include any quadratic or interaction terms. Instead, we will consider all individual candidate predictors in the dataset except `quality`; as we previously discussed, `quality` is a subjecting wine rating that may be difficult to obtain for future wines where we want to predict type.

Let's tune the model, considering 100 candidates for the regularization hyperparameter, denoted `C`. Note that we are using the `liblinear` algorithm, which, per the documentation for `LogisticRegressionCV`, is one of only two options for a model with an L1 penalty.

In [ ]:
# Using 10-fold CV to optimize LR with L1 penalty
lr_l1_model = LogisticRegressionCV(cv = 10,
                              solver = "liblinear",
                              Cs = 100,
                              penalty = "l1",
                              scoring = "neg_log_loss",
                              random_state = 5) \
                              .fit(X_train_c.drop("quality", axis = 1), y_train_c)

# Extracting the optimal value for C
print(lr_l1_model.C_)


Now that we have the best `C`, we can fit a model to the full training set.

In [ ]:
# Fitting best LR model with L1 penalty to full training set
lr_l1_best = LogisticRegression(solver = "liblinear", penalty = "l1",
                             random_state = 5, C = lr_l1_model.C_[0]) \
                         .fit(X_train_c.drop("quality", axis = 1), y_train_c)

We now have a tuned logistic regression model with L1 penalty that is ready to predict wine type for the full test set.

#### Tuning Logistic Regression with L2 Penalty

To tune the logistic regression model with an L2 penalty, we will use the same process and design matrix we used for the L1-penalty model. We will use the same `LogisticRegressionCV` function, which also denotes the L2-penalty model's hyperparamter `C`. The similarities are borderline confusing....

Alas, let's tune the model.

In [ ]:
# Using 10-fold CV to optimize LR with L2 penalty
lr_l2_model = LogisticRegressionCV(cv = 10,
                              solver = "liblinear",
                              Cs = 100,
                              penalty = "l2",
                              scoring = "neg_log_loss",
                              random_state = 5) \
                              .fit(X_train_c.drop("quality", axis = 1), y_train_c)

# Extracting the optimal value for C
print(lr_l2_model.C_)

Now that we have the best hyperparameter value for predicting wine type, at least based on these cross-validation results, let's fit a model to the full training set.

In [ ]:
# Fitting best LR model with L2 penalty to full training set
lr_l2_best = LogisticRegression(solver = "liblinear", penalty = "l2",
                             random_state = 5, C = lr_l2_model.C_[0]) \
                         .fit(X_train_c.drop("quality", axis = 1), y_train_c)

We have now fit the best models for three of the four considered model classes to the full training set. One more class to go!

#### Tuning Logistic Regression with Elastic Net Penalty

The final model class we will consider is the logistic regression model with an elastic net penalty. That is, a logistic regression model with both an L1 and an L2 penalty.

To tune this model, we will again use `LogisticRegressionCV`, but this time we will consider both candidate `C` values and candidate `l1_ratio` values. For `C`, we'll consider 50 function-chosen candidates. For `l1_ratio`, we'll consider 10 candidates ranging from 0 to 1. These combine to determine the relative scale of the L1 and L2 penalties. For the elastic net model, we must use the `saga` algorithm to solve.

In [ ]:
# Using 10-fold CV to optimize LR with elastic net penalty
lr_en_model = LogisticRegressionCV(cv = 10,
                              solver = "saga",
                              Cs = 50,
                              l1_ratios = [0.01, 0.05, 0.1, 0.25, 0.5, 0.75,
                                           0.9, 0.95, 0.99, 1],
                              penalty = "elasticnet",
                              scoring = "neg_log_loss",
                              random_state = 5,
                                   max_iter = 1000) \
                              .fit(X_train_c.drop("quality", axis = 1), y_train_c)

# Extracting the optimal value for C
print(lr_en_model.C_, lr_en_model.l1_ratio_)

To the reader, be thankful you didn't need to wait for the previous code block to run.... Focusing on the results, the `l1_ratio` of 1 indicates we have effectively obtained the L1-penalty results; the slightly different value of `C` for this model and the tuned L1-penalty model above is likely due to different numbers of candidate `C`as well as the different algorithm.

Regardless, we now have the best combination of hyperparmeters, so we can fit an elastic net-penalty model to the full training set.

In [ ]:
# Fitting best LR model with elastic net penalty to full training set
lr_en_best = LogisticRegression(solver = "saga", penalty = "elasticnet",
                             random_state = 5, C = lr_en_model.C_[0],
                                l1_ratio = lr_en_model.l1_ratio_[0],
                                max_iter = 1000) \
                         .fit(X_train_c.drop("quality", axis = 1), y_train_c)

We now have the best model from each model class fit to the full training set. Let's predict the test set!

#### Choosing a Final Model to Predict Wine Type

Let's use our four "best" models to predict wine type for the test set, as this will indicate which model we will fit to the full dataset and use for future predictions.

Again, we will consider both log-loss and accuracy when making our decision. Note that to calculate log-loss, we need to obtain the probability of each wine in the test set being red in addition to the corresponding type prediction.

In [ ]:
# Predicting with LR model
lr_pred = lr_best.predict(PolynomialFeatures(interaction_only=False,
                                              include_bias = False) \
                        .fit_transform(X_test_c[["alcohol", "sulphates",
                                                  "residual_sugar"]]))

# Obtaining class probabilities for LR model
lr_probs = lr_best.predict_proba(PolynomialFeatures(interaction_only=False,
                                              include_bias = False) \
                        .fit_transform(X_test_c[["alcohol", "sulphates",
                                                  "residual_sugar"]]))

# Predicting with LR L1 model
lr_l1_pred = lr_l1_best.predict(X_test_c.drop("quality", axis = 1))

# Obtaining class probabilities for LR L1 model
lr_l1_probs = lr_l1_best.predict_proba(X_test_c.drop("quality", axis = 1))

# Predicting with LR L2 model
lr_l2_pred = lr_l2_best.predict(X_test_c.drop("quality", axis = 1))

# Obtaining class probabilities for LR L2 model
lr_l2_probs = lr_l2_best.predict_proba(X_test_c.drop("quality", axis = 1))

# Predicting with LR Elastic Net model
lr_en_pred = lr_en_best.predict(X_test_c.drop("quality", axis = 1))

# Obtaining class probabilities for LR Elastic Net model
lr_en_probs = lr_en_best.predict_proba(X_test_c.drop("quality", axis = 1))

# Printing log-loss
print("LR log-loss:", log_loss(y_test_c, lr_probs),
      "\nLR L1 log-loss:", log_loss(y_test_c, lr_l1_probs),
      "\nLR L2 log-loss:", log_loss(y_test_c, lr_l2_probs),
      "\nElastic Net log-loss:", log_loss(y_test_c, lr_en_probs))

# Printing accuracy
print("\nLR accuracy:", accuracy_score(y_test_c, lr_pred),
      "\nLR L1 accuracy:", accuracy_score(y_test_c, lr_l1_pred),
      "\nLR L2 accuracy:", accuracy_score(y_test_c, lr_l2_pred),
      "\nElastic Net accuracy:", accuracy_score(y_test_c, lr_en_pred))

For the classification problem, the tuned logistic regression with an L1 penalty and the tune logistic regression with an elastic net penalty produce nearly identical results. In fact, the two models produce identical accuracy, while the L1-penalty model only marginally outperforms the elastic net-penalty model in test log-loss. This is not surprising as the best models from each class are effectively identical.

Based on these results, we select the logistic regression with an L1 penalty as our best overall model. However, all three regularized models produce very good results. All three have test accuracies above 99.6\%!!!